In [0]:
from pyspark.sql import functions as F

# Read Bronze streaming table
events = (
    spark.readStream
    .table("workspace.bronze.web_events")
)

# Convert the raw timestamp string into a proper TIMESTAMP.
# The source data contains more than one timestamp format,
# so we safely try both formats.
events = (
    events
    .withColumn(
        "event_time",
        F.coalesce(
            F.expr("try_to_timestamp(event_timestamp, 'dd-MM-yyyy HH:mm')"),
            F.expr("try_to_timestamp(event_timestamp, 'yyyy-MM-dd HH:mm:ss')")
        )
    )
)

# Keep only records where the timestamp was successfully parsed
events = events.filter(F.col("event_time").isNotNull())

# 1-hour streaming aggregation
analytics = (
    events
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        F.window("event_time", "1 hour"),
        "event_type"
    )
    .count()
)

query = (
    analytics.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/ecommerce_checkpoints/event_analytics_v3"
    )
    .trigger(availableNow=True)
    .toTable("workspace.gold.web_event_analytics")
)

query.awaitTermination()

In [0]:
%sql
SELECT *
FROM workspace.gold.web_event_analytics
ORDER BY window.start, event_type;

In [0]:
%sql
SELECT
    window.start AS window_start,
    window.end AS window_end,
    event_type,
    count
FROM workspace.gold.web_event_analytics
ORDER BY window.start, event_type;